# IndexTTS2 - 部署笔记本 (Colab/Kaggle)\n\n**部署流程**: ①安装mamba → ②创建Python 3.10环境 → ③安装依赖 → ④克隆代码

In [ ]:
# ============================================\n# Cell 1: 环境准备 (3步流程)\n# ============================================\nimport os, sys, json, subprocess, urllib.request\n\n# 基础配置\nIN_COLAB = 'google.colab' in sys.modules\nIN_KAGGLE = os.path.exists('/kaggle/input')\nWORK_DIR = '/content' if IN_COLAB else '/kaggle/working' if IN_KAGGLE else '/tmp'\nREPO_DIR = f"{WORK_DIR}/index-tts"\nMAMBA_ROOT = f"{WORK_DIR}/mamba"\nMAMBA_ENV = f"{MAMBA_ROOT}/envs/tts"\n\nprint(f"🔍 环境: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Other'}")\nprint(f"📁 工作目录: {WORK_DIR}")\nprint("\n" + "="*60)\n\n# =====================================================\n# 第1步: 安装mamba (必须成功)\n# =====================================================\nprint("\n📦 第1步: 安装mamba...")\n\nMICROMAMBA_BIN = None\n\n# 1.1 尝试使用现有的mamba (必须能执行create命令)\ndef check_existing_mamba():\n    """检查系统中是否有真正的mamba包管理器 (不是测试框架)"""\n    try:\n        result = subprocess.run(['which', 'mamba'], capture_output=True, text=True)\n        if result.returncode == 0:\n            bin_path = result.stdout.strip()\n            # 关键测试：执行 create --help 看是否支持包管理功能\n            test_result = subprocess.run([bin_path, 'create', '--help'], \n                                        capture_output=True, text=True, timeout=5)\n            # 真正的mamba会显示create命令的帮助，测试框架会报错\n            if test_result.returncode == 0 and 'environment' in test_result.stdout.lower():\n                print(f"✅ 发现真正的mamba包管理器: {bin_path}")\n                return bin_path\n            else:\n                print(f"⚠️ {bin_path} 不是包管理器 (可能是测试框架)，跳过")\n    except Exception as e:\n        pass\n    return None\n\nMICROMAMBA_BIN = check_existing_mamba()\n\n# 1.2 如果没有，安装micromamba\nif not MICROMAMBA_BIN:\n    print("🔧 安装 micromamba...")\n    \n    os.environ['MAMBA_ROOT_PREFIX'] = MAMBA_ROOT\n    \n    install_script = f"{WORK_DIR}/install.sh"\n    urllib.request.urlretrieve('https://micro.mamba.pm/install.sh', install_script)\n    os.chmod(install_script, 0o755)\n    \n    subprocess.run(['bash', install_script], check=True)\n    \n    # 查找micromamba\n    for path in [\n        '/root/.local/bin/micromamba',\n        f"{os.environ.get('HOME', '/root')}/.local/bin/micromamba",\n        f"{MAMBA_ROOT}/bin/micromamba"\n    ]:\n        if os.path.exists(path):\n            MICROMAMBA_BIN = path\n            print(f"✅ 安装成功: {path}")\n            break\n    \n    if not MICROMAMBA_BIN:\n        raise RuntimeError("❌ mamba安装失败，无法继续")\n\n# 1.3 验证并添加到PATH\nbin_dir = os.path.dirname(MICROMAMBA_BIN)\nos.environ['PATH'] = f"{bin_dir}:{os.environ['PATH']}"\n\n# 创建mamba别名\nmamba_link = os.path.join(bin_dir, 'mamba')\nif not os.path.exists(mamba_link):\n    os.symlink(MICROMAMBA_BIN, mamba_link)\n\nprint(f"✅ mamba就绪: {MICROMAMBA_BIN}")\n\n# =====================================================\n# 第2步: 创建Python 3.10环境 (必须成功)\n# =====================================================\nprint("\n📦 第2步: 创建Python 3.10环境...")\nprint("⏳ 安装python=3.10和cudatoolkit=11.8，请等待...")\n\n# 尝试创建带CUDA的环境\ncmd = f"{MICROMAMBA_BIN} create -y -p {MAMBA_ENV} python=3.10 cudatoolkit=11.8 -c conda-forge -c nvidia"\nresult = subprocess.run(cmd, shell=True, capture_output=True, text=True)\n\nif result.returncode != 0:\n    print("⚠️ 带CUDA的环境创建失败，尝试仅安装Python...")\n    # Fallback: 只安装python\n    cmd = f"{MICROMAMBA_BIN} create -y -p {MAMBA_ENV} python=3.10 -c conda-forge"\n    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)\n    \n    if result.returncode != 0:\n        print(f"❌ 环境创建失败:")\n        print(result.stderr)\n        raise RuntimeError("环境创建失败")\n    else:\n        print("✅ Python 3.10环境创建成功 (无CUDA)")\n\nprint("✅ Python 3.10环境创建成功")\n\n# 设置Python和pip路径\nMAMBA_PYTHON = f"{MAMBA_ENV}/bin/python"\nMAMBA_PIP = f"{MAMBA_ENV}/bin/pip"\n\nprint(f"   Python: {MAMBA_PYTHON}")\nprint(f"   pip: {MAMBA_PIP}")\n\n# =====================================================\n# 第3步: 安装依赖 (使用mamba环境的pip)\n# =====================================================\nprint("\n📦 第3步: 安装Python依赖...")\n\n# 3.1 PyTorch (项目要求版本)\nprint("   安装 PyTorch (尝试cu128)...")\ntry:\n    # 首先尝试项目要求的cu128\n    subprocess.run([\n        MAMBA_PIP, 'install', '-q',\n        'torch==2.8.0', 'torchaudio==2.8.0',\n        '--index-url', 'https://download.pytorch.org/whl/cu128'\n    ], check=True)\n    print("   ✅ PyTorch cu128 安装成功")\nexcept:\n    print("   ⚠️ cu128不可用，尝试cu121...")\n    try:\n        # 回退到cu121\n        subprocess.run([\n            MAMBA_PIP, 'install', '-q',\n            'torch==2.5.1', 'torchaudio==2.5.1',\n            '--index-url', 'https://download.pytorch.org/whl/cu121'\n        ], check=True)\n        print("   ✅ PyTorch cu121 (2.5.1) 安装成功")\n    except:\n        print("   ⚠️ CUDA版本均不可用，使用标准pip安装...")\n        # 最后回退到标准pip（无CUDA指定）\n        subprocess.run([MAMBA_PIP, 'install', '-q', 'torch', 'torchaudio'], check=True)\n        print("   ✅ PyTorch (标准版) 安装成功")\n\n# 3.2 基础依赖\nprint("   安装基础依赖...")\ndeps = ["pyngrok", "gradio==5.45.0", "fastapi", "uvicorn", "python-multipart", "huggingface-hub", "git-lfs", "requests"]\nfor dep in deps:\n    subprocess.run([MAMBA_PIP, 'install', '-q', dep], check=True)\n\nprint("✅ 依赖安装完成")\n\n# =====================================================\n# 第4步: 克隆代码\n# =====================================================\nprint("\n📦 第4步: 克隆代码...")\n\nif os.path.exists(REPO_DIR):\n    subprocess.run(['rm', '-rf', REPO_DIR], check=True)\n\n# 跳过LFS文件\nenv = os.environ.copy()\nenv['GIT_LFS_SKIP_SMUDGE'] = '1'\nsubprocess.run([\n    'git', 'clone', '--depth', '1', '-b', 'dev',\n    'https://github.com/infinite-gaming-studio/index-tts.git',\n    REPO_DIR\n], env=env, check=True)\n\nprint(f"✅ 代码克隆完成: {REPO_DIR}")\n\n# =====================================================\n# 保存配置\n# =====================================================\nconfig = {\n    'work_dir': WORK_DIR,\n    'repo_dir': REPO_DIR,\n    'mamba_env': MAMBA_ENV,\n    'python': MAMBA_PYTHON,\n    'pip': MAMBA_PIP,\n    'in_colab': IN_COLAB,\n    'in_kaggle': IN_KAGGLE\n}\n\nwith open('/tmp/notebook_config.json', 'w') as f:\n    json.dump(config, f, indent=2)\n\nprint("\n" + "="*60)\nprint("✅ Cell 1 完成!")\nprint(f"   Python: {MAMBA_PYTHON}")\nprint(f"   代码: {REPO_DIR}")\nprint(f"\n👉 请重新启动kernel，然后运行Cell 2")\nprint(f"   (使用: {MAMBA_PYTHON})")\nprint("="*60)

In [ ]:
# ============================================\n# Cell 2: 安装项目依赖和模型\n# 确保使用mamba环境的Python运行此Cell\n# ============================================\n\nimport sys, json, subprocess, os\n\n# 加载配置\nwith open('/tmp/notebook_config.json', 'r') as f:\n    config = json.load(f)\n\nREPO_DIR = config['repo_dir']\nPYTHON = config['python']\nPIP = config['pip']\n\nprint(f"✅ 使用mamba Python: {PYTHON}")\n\n# 添加项目路径\nsys.path.insert(0, REPO_DIR)\n\n# 安装项目依赖（跳过PyTorch，Cell 1已安装）\nprint("\n📦 安装项目依赖（跳过PyTorch）...")\n\nfrom deploy.utils import DependencyInstaller, ModelDownloader, check_model_exists\n\n# 只安装核心依赖，跳过PyTorch（批量安装更快）\nsubprocess.run([PIP, "install", "-q"] + DependencyInstaller.CORE_DEPS, check=True)\nprint("✅ 核心依赖安装完成")\n\n# 安装项目\nprint("\n📦 安装项目...")\nsubprocess.run([PIP, 'install', '-q', '-e', REPO_DIR], check=True)\n\n# 下载模型\nprint("\n🤖 检查模型...")\nCHECKPOINT_DIR = f"{REPO_DIR}/checkpoints"\n\n# 统一检查逻辑：config.yaml + 模型权重文件\ndef check_model_complete(checkpoint_dir):\n    """检查模型是否完整（配置文件+权重文件）"""\n    config_file = os.path.join(checkpoint_dir, "config.yaml")\n    if not os.path.exists(config_file):\n        return False, "配置文件不存在"\n    \n    if not os.path.isdir(checkpoint_dir):\n        return False, "检查点目录不存在"\n    \n    files = os.listdir(checkpoint_dir)\n    model_files = [f for f in files if f.endswith(('.pt', '.bin', '.pth', '.safetensors', '.ckpt'))]\n    \n    if not model_files:\n        return False, f"无权重文件，现有文件: {files}"\n    \n    return True, f"找到 {len(model_files)} 个权重文件"\n\nis_complete, msg = check_model_complete(CHECKPOINT_DIR)\nprint(f"   检查结果: {msg}")\n\nif not is_complete:\n    print("   模型不完整，开始下载...")\n    ModelDownloader.download(CHECKPOINT_DIR)\n    # 下载后再次检查\n    is_complete, msg = check_model_complete(CHECKPOINT_DIR)\n    print(f"   下载后检查: {msg}")\nelse:\n    print(f"   ✅ 模型已完整存在")\n\nprint("\n" + "="*60)\nprint("✅ Cell 2 完成! 可以启动服务了")\nprint("="*60)